In [0]:
CATALOG = "workspace"
SCHEMA = "bronze"
BASE = f'/Volumes/{CATALOG}/{SCHEMA}/landing'

spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')

import pyspark.sql.functions as F


In [0]:
file_list = {
    "trades" : "trades_raw",
    "ledger_entries" : "ledger_raw",
    "reference/counterparties" : "counterparties_raw",
    "reference/instruments" : "instruments_raw",
    "reference/gl_accounts" : "gl_accounts"
}


for path, table in file_list.items():
    df = ( 
          spark.read.format("csv")
          .option("header", True)
          .load(f"{BASE}/{path}/")
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingest_time", F.current_timestamp())
    )
    
    (df.write.mode("overwrite").format("delta")
     .option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")
     )
    
    print(f'Loaded table {table}; with {df.count()} records' )

In [0]:
display(spark.sql(f"""select * from {CATALOG}.{SCHEMA}.trades_raw"""))

In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .load(f"{BASE}/trades/")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingest_time", F.current_timestamp())
)

df.printSchema()



df_01 = (
    spark.read.format("csv")
    .option("header", True)
    .load(f"{BASE}/trades/")
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingest_time", F.current_timestamp())
)
    

In [0]:
print (f'Total count - {df.count()}; distinct trades - {df.select("trade_id").distinct().count()}')

print(f'Total count - {df.count()}\
    disctinct_tades - {df.select(F.col("trade_id")).distinct().count()}')

In [0]:
df.select(F.col("trade_id")).distinct().count()

In [0]:
df_casted =(
    df.withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("price",F.col("price").cast("decimal(18,4)"))
    .withColumn("trade_ts", F.col("trade_ts").cast("timestamp"))
)



df_casted_01 = (
    df.withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("price", F.col("price").cast("decimal(18,4)"))
    .withColumn("trade_ts", F.col("trade_ts").cast("timestamp"))
)

In [0]:
df_casted.show(10)

In [0]:
df_casted.printSchema()

In [0]:
#Tast 2 check casting string to int and fix it 
#Task 2.1
data = ["12x"]
df_test = spark.createDataFrame(data, schema = ["header"])

df_test_casted = df_test.withColumn("header", F.col("header").cast("int"))

df_test_casted.show()

In [0]:
#Task 2.2 - fix using try_cast

df_test_casted_fix = df_test.withColumn("header", F.col("header").try_cast("int"))

df_test_casted_fix.show()

In [0]:
df_casted.printSchema()

df_casted_trans = (
    df_casted.withColumn("notional", F.round(F.col("quantity") * F.col("price"), 2))
)

#df_casted_trans.count()

df_top3 = df_casted_trans.sort(F.col("notional").desc()).select("trade_id", "notional").limit(3)

df_top3.printSchema()

In [0]:
#Task 4 - calculate quantity signage based on Side

df_casted_qty = (
    df_casted.withColumn("signed_qty", 
                         F.when(F.col("side").eqNullSafe("SELL"), F.col("quantity") * -1 )
                         .when(F.col("side").eqNullSafe("BUY"), F.col("quantity")) )
)

df_signed_sum = (
    df_casted.agg(F.sum( 
                        F.when( F.col("side").eqNullSafe("BUY"), F.col("quantity") )
                        .when(F.col("side").eqNullSafe("SELL"), F.col("quantity") * -1) ).alias("Total Signed Sum"))
)

df_signed_sum.show()

print(
    f'Sum of signed qty : {df_casted_qty.agg(F.sum(F.col("signed_qty"))).collect()[0][0]}'
)

In [0]:
#Task 5

print(
    f'Count of XXX Currency - {df_casted.filter(F.col("currency") == 'XXX').count()}\
    \nCount of negative Quantity - {df_casted.filter(F.col("quantity") < 0).count()}\
    \nCount of BUY - {df_casted.filter((F.col("side") == "BUY") & (F.col("quantity") > 5000)).count()}'
)

In [0]:
print(f'C0001 count - \
    {df_casted_qty.filter(F.col("counterparty_id") == "C0001").count()}\
        \nNon C0001 count \
    {df_casted_qty.filter(F.col("counterparty_id") != "C0001").count()}\
        \nNull counter party count - \
    {df_casted_qty.filter(F.col("counterparty_id").isNull()).count()}\
        \nTotal Count - {df_casted_qty.count()}'
    )
print (f'Total count - {df_casted_qty.filter(F.col("counterparty_id") == "C0001").count() + df_casted_qty.filter(F.col("counterparty_id") != "C0001").count()} ')

In [0]:
print (f'Count of C0001 - \
    {df_casted_qty.filter(  F.col("counterparty_id") == "C0001" ).count()}')

print (f'Count of Non C0001 - \
    {df_casted_qty.filter( ~ F.col("counterparty_id").eqNullSafe("C0001")).count()}')


print (f'Count of NON C0001 - \
    {df_casted_qty.filter( ( F.col("counterparty_id") != "C0001" ) | F.col("counterparty_id").isNull() ).count() }')


In [0]:
df_us_active = df_casted_trans.filter((F.col("currency") == "USD") & (F.col("status") != "CANCEL"))

#Correct version
df_us_active_01 = df_casted_trans.filter((F.col("currency") == "USD") & ~( F.col("status").eqNullSafe("CANCEL") ))


df_us_active.select(F.sum("notional").alias("total_notional")).show()


df_us_active_01.agg(F.sum("notional").alias("total")).show()
